# Homework 2
## Module 2: Vector Search
### LLM Zoomcamp 2026

Author: Debabrata Mishra

---

## Objective

This homework demonstrates the implementation of semantic search using vector embeddings.

The tasks cover:

- Embedding generation
- Cosine similarity
- Vector search
- Chunking
- Vector indexing
- Keyword search
- Hybrid Search (Reciprocal Rank Fusion)

Unlike Homework 1 (Agentic RAG), this homework focuses entirely on retrieval.

# Learning Objectives

After completing this homework I should understand

1. What embeddings are

2. How embedding models work

3. Cosine similarity

4. Vector search

5. Chunking

6. Semantic retrieval

7. Difference between keyword and vector search

8. Hybrid search

9. Reciprocal Rank Fusion (RRF)

# Homework Questions

Q1 Embedding a Query

Q2 Cosine Similarity

Q3 Chunking and Manual Vector Search

Q4 Vector Search using MinSearch

Q5 Text Search vs Vector Search

Q6 Hybrid Search using Reciprocal Rank Fusion

# Environment Setup
# Imports

In [12]:
import os
import sys

print("Current working directory:")
print(os.getcwd())

print("\nPython executable:")
print(sys.executable)

Current working directory:
C:\Users\dmish\llm-zoomcamp-code\homework_02_vector_search

Python executable:
C:\Users\dmish\llm-zoomcamp-code\.venv\Scripts\python.exe


In [13]:
import os
import sys

# Get the absolute path of the project root
project_root = os.path.abspath("..")

# Add it to Python's module search path
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(project_root)

C:\Users\dmish\llm-zoomcamp-code


In [14]:
import numpy as np

from tqdm import tqdm

from gitsource import GithubRepositoryDataReader
from gitsource import chunk_documents

from embedder import Embedder

import minsearch

In [15]:
# ============================================================
# Project Setup (Run once after restarting the kernel)
# ============================================================

import os
import sys

# ------------------------------------------------------------
# Project root directory
# Current notebook:
# C:\Users\dmish\llm-zoomcamp-code\homework_02_vector_search
#
# Project root:
# C:\Users\dmish\llm-zoomcamp-code
# ------------------------------------------------------------

PROJECT_ROOT = os.path.abspath("..")

print(f"Project Root : {PROJECT_ROOT}")

# ------------------------------------------------------------
# Add project root to Python search path
# Enables importing:
#   embedder.py
#   download.py
#   rag_helper.py
# and other shared modules.
# ------------------------------------------------------------

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root added to sys.path")

# ------------------------------------------------------------
# Define ONNX model location
# ------------------------------------------------------------

MODEL_PATH = os.path.join(
    PROJECT_ROOT,
    "models",
    "Xenova",
    "all-MiniLM-L6-v2"
)

print(f"Model Path : {MODEL_PATH}")

# ------------------------------------------------------------
# Verify the model exists
# ------------------------------------------------------------

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model directory not found:\n{MODEL_PATH}\n\n"
        "Run:\npython download.py"
    )

print("Model directory found.")

# ------------------------------------------------------------
# Import the shared embedder
# ------------------------------------------------------------

from embedder import Embedder

# ------------------------------------------------------------
# Create ONE embedder instance for the notebook
# ------------------------------------------------------------

embedder = Embedder(path=MODEL_PATH)

print("Embedder loaded successfully!")

# ------------------------------------------------------------
# Display Python information
# ------------------------------------------------------------

print("\nPython Executable:")
print(sys.executable)

print("\nPython Version:")
print(sys.version)

Project Root : C:\Users\dmish\llm-zoomcamp-code
Project root added to sys.path
Model Path : C:\Users\dmish\llm-zoomcamp-code\models\Xenova\all-MiniLM-L6-v2
Model directory found.
Embedder loaded successfully!

Python Executable:
C:\Users\dmish\llm-zoomcamp-code\.venv\Scripts\python.exe

Python Version:
3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


In [16]:
import sys

print(sys.executable)
print(sys.version)

from embedder import Embedder

embedder = Embedder(
    path="../models/Xenova/all-MiniLM-L6-v2"
)

print("Success!")

C:\Users\dmish\llm-zoomcamp-code\.venv\Scripts\python.exe
3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Success!


# Load Embedding Model

# Load the Course Lesson Dataset

The knowledge base consists of the 72 lesson pages from the
DataTalksClub LLM Zoomcamp GitHub repository.

We load the markdown files directly from GitHub using
GithubRepositoryDataReader and pin to commit 8c1834d to ensure
everyone works with the same dataset.

In [24]:
# Configure the GitHub repository reader
# This will retrieve all lesson markdown files
# from the pinned course commit (8c1834d).

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

In [25]:
# Download and parse the lesson pages
documents = [file.parse() for file in reader.read()]

In [26]:
# Display the total number of lesson pages
len(documents)

72

In [27]:
# Inspect the first lesson page
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [28]:
# Display the filename of the first lesson
documents[0]["filename"]

'01-agentic-rag/lessons/01-intro.md'

In [29]:
# Display the first 500 characters of the lesson content
print(documents[0]["content"][:600])

# Introduction

Video: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In this module, we'll build a working Retrieval-Augmented
Generation (RAG) system from scratch, step by step.

We write everything in plain Python. We build a small search index by
hand and call the LLM ourselves. I want you to see every piece first.
That way you know what a framework does for you before you reach for
one.

Places where you can find me:

- [My substack](https://alexeyondata.substack.com/)
- [LinkedIn](https://www.linkedin.com/in/agrigorev/)
- [X](htt


In [30]:
query = "How does approximate nearest neighbor search work?"

v = embedder.encode(query)

print(v.shape)
print(v[0])

(384,)
-0.02058203437252893


### Interpretation

The ONNX embedding model converts the input query into a dense numerical
representation called an embedding.

The output vector has 384 dimensions, where each element captures part of
the semantic meaning of the sentence.

The first element of the embedding vector is:

-0.02058203437252893

This value itself has no human-readable meaning. It is one coordinate in a
384-dimensional semantic space learned during model training.

When compared with embeddings of other documents using cosine similarity,
the entire vector—not any individual value—is used to measure semantic
similarity.

Therefore, the correct homework answer is:

**-0.02**

# Q2. Cosine Similarity

## Objective

Compute the cosine similarity between

- the query embedding from Q1

and

- the lesson page

02-vector-search/lessons/07-sqlitesearch-vector.md

Since the ONNX embedder returns normalized vectors, the dot product
between two vectors is equal to their cosine similarity.

In [31]:
# Find the required lesson page

target_doc = None

for doc in documents:
    if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md":
        target_doc = doc
        break

print(target_doc["filename"])

02-vector-search/lessons/07-sqlitesearch-vector.md


In [32]:
# Embed the lesson page

doc_vector = embedder.encode(target_doc["content"])

print(doc_vector.shape)

(384,)


In [33]:
# Since both vectors are normalized,
# dot product == cosine similarity

similarity = np.dot(v, doc_vector)

print(similarity)

0.36107026789538205


## Interpretation

The lesson page and the query were converted into 384-dimensional
normalized embedding vectors.

Since both vectors are normalized, their dot product equals their cosine
similarity.

The computed similarity is:

0.36107026789538205

This indicates a moderate semantic similarity between the user's query
and the lesson page.

Although the lesson discusses vector search concepts, it is not an
exact explanation of approximate nearest neighbor search, so the
similarity is well below 1.0.

Therefore, the correct homework answer is:

**0.37**

# Q3. Chunking and Search by Hand

## Objective

Instead of embedding an entire lesson page, split each page into
overlapping chunks.

Each chunk is embedded separately.

The query embedding is then compared against every chunk embedding,
and the chunk with the highest cosine similarity is returned.

This demonstrates how semantic vector search works internally before
using libraries such as MinSearch.

In [34]:
# Split lesson pages into overlapping chunks

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

print(f"Total chunks: {len(chunks)}")

Total chunks: 295


In [35]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [36]:
# Extract only the text from each chunk

chunk_texts = [
    chunk["content"]
    for chunk in chunks
]

len(chunk_texts)

295

In [37]:
# Generate embeddings for all chunks

X = embedder.encode_batch(chunk_texts)

print(X.shape)

(295, 384)


In [38]:
# Compute similarity between the query and every chunk

scores = X.dot(v)

print(scores.shape)

(295,)


In [39]:
best_idx = np.argmax(scores)

best_idx

np.int64(94)

In [40]:
best_chunk = chunks[best_idx]

best_chunk["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [41]:
scores[best_idx]

np.float64(0.6489016436447387)

In [42]:
print(best_chunk["content"][:1200])

rch. We score
the query against every document and pick the top ones. It always finds
the true top matches, but it pays for that by touching everything.

Approximate nearest neighbor (ANN) search takes a shortcut. Instead of
comparing against everything, it first narrows down to a region of
likely matches. Then it scores only within that region. It may miss the
absolute best match, but the results are still good and it's much
faster.

```text
NN (exact):    compare query against ALL documents -> top 5
ANN (approx):  narrow down to a region -> compare within region -> top 5
```

## sqlitesearch

sqlitesearch is the persistent sibling of minsearch, and it solves both
problems at once.

We already used it in module 1 for persistent text search. It also does
vector search through its `VectorSearchIndex` class. It stores vectors
in SQLite, a real on-disk database, and uses ANN strategies for
retrieval. Because the data lives on disk, one process can write the
vectors and another can read th

In [43]:
top5 = np.argsort(scores)[::-1][:5]

for rank, idx in enumerate(top5, start=1):
    print(f"{rank}. {chunks[idx]['filename']}")
    print(f"   Score: {scores[idx]:.4f}")
    print()

1. 02-vector-search/lessons/07-sqlitesearch-vector.md
   Score: 0.6489

2. 01-agentic-rag/lessons/05-search.md
   Score: 0.5510

3. 04-evaluation/lessons/05-search-metrics.md
   Score: 0.4066

4. 02-vector-search/lessons/04-vector-search.md
   Score: 0.4062

5. 06-best-practices/lessons/03-reranking.md
   Score: 0.4061



## Interpretation of Results

### Results

- Total lesson pages: **72**
- Total chunks created: **295**
- Embedding dimension: **384**
- Highest cosine similarity: **0.6489**
- Best matching lesson:

```
02-vector-search/lessons/07-sqlitesearch-vector.md
```

---

### Why Chunking Improves Retrieval

Embedding an entire lesson page compresses multiple topics into a single
vector representation.

For example, one lesson may contain:

- Introduction
- Theory
- Examples
- Installation
- Code
- Best practices

When all these topics are embedded together, the resulting vector represents
the **average semantic meaning** of the page. Consequently, the similarity to
a focused query may be lower.

Chunking addresses this problem by splitting each lesson into smaller,
overlapping sections.

Each chunk therefore captures a much more focused semantic topic, producing a
more discriminative embedding.

As a result, the query is matched against the specific section discussing the
topic of interest rather than against an entire lesson covering multiple
subjects.

---

### Comparison with Q2

| Question | Retrieval Unit | Highest Similarity |
|-----------|----------------|-------------------:|
| Q2 | Entire lesson page | **0.3611** |
| Q3 | Best matching chunk | **0.6489** |

The similarity score increased substantially because the retrieved chunk is
focused almost entirely on **Approximate Nearest Neighbour (ANN) Search**,
whereas the complete lesson page also contains introductory material, SQLite
configuration, code examples and other concepts that dilute the embedding.

---

### Key Learning

This exercise illustrates why virtually all modern Retrieval-Augmented
Generation (RAG) systems perform **document chunking before indexing**.

Rather than retrieving entire documents, vector databases retrieve the
highest-ranking chunks, providing the language model with much more focused
and relevant context.

This improves retrieval quality while simultaneously reducing unnecessary
context supplied to the Large Language Model.

## From Embeddings to Semantic Search

The first three questions of this homework build the foundations of a vector
search engine in a gradual manner.

### Q1 – Embedding a Query

The user's natural language query was converted into a dense numerical
representation (embedding) using the ONNX version of the
**all-MiniLM-L6-v2** model.

```
Query
   │
   ▼
Embedding Model
   │
   ▼
384-dimensional Vector
```

---

### Q2 – Measuring Semantic Similarity

The query embedding was compared with the embedding of a single lesson page
using **cosine similarity**.

Because the embedding model produces **normalized vectors**, the cosine
similarity is simply the **dot product** between the two vectors.

This allowed us to measure how semantically related the query is to one
specific document.

---

### Q3 – Manual Vector Search

Instead of comparing the query against only one document, we now compare it
against **every document chunk**.

Each lesson page is first divided into overlapping chunks. Every chunk is
embedded independently and compared with the query embedding.

The chunk with the highest cosine similarity is returned as the most relevant
piece of information.

```
User Query
      │
      ▼
Query Embedding
      │
      ▼
Compare with
295 Chunk Embeddings
      │
      ▼
295 Similarity Scores
      │
      ▼
Highest Scoring Chunk
      │
      ▼
Most Relevant Result
```

This manual implementation demonstrates the core retrieval mechanism behind
modern vector search systems before introducing dedicated libraries such as
**MinSearch** in the next question.

> **Key Takeaway**
>
> Q1 taught us how to represent text numerically using embeddings.
>
> Q2 showed how embeddings can be compared using cosine similarity.
>
> Q3 combined these concepts to build a complete semantic retrieval pipeline,
> demonstrating the fundamental principle behind vector search engines used in
> modern RAG systems.

Query Embedding
       │
       ▼
295 Chunk Embeddings
       │
       ▼
Dot Product
       │
       ▼
Sort Scores
       │
       ▼
Return Top Result

# Q4. Vector Search with MinSearch

## Objective

In Question 3, we implemented semantic vector search manually by:

- embedding every document chunk,
- computing cosine similarity,
- ranking the similarity scores,
- retrieving the highest-scoring chunk.

Although this helped us understand the underlying mathematics, real-world
applications rarely implement vector search manually.

Instead, production systems use specialised search libraries that handle:

- vector indexing,
- similarity computation,
- efficient retrieval,
- ranking.

In this question, we replace our manual implementation with the
**VectorSearch** class from the **MinSearch** library while using the same
embeddings generated previously.

In [45]:
import minsearch
print(minsearch.__version__)

import inspect
print(inspect.signature(minsearch.VectorSearch))

print(dir(minsearch))

0.1.0
(keyword_fields=None, numeric_fields=None, date_fields=None)
['AppendableIndex', 'DEFAULT_ENGLISH_STOP_WORDS', 'Highlighter', 'Index', 'STEMMERS', 'Tokenizer', 'VectorSearch', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'append', 'filters', 'get_stemmer', 'highlighter', 'lancaster_stemmer', 'minsearch', 'porter_stemmer', 'snowball_stemmer', 'stemmers', 'tokenizer', 'vector']


In [46]:
import inspect
import minsearch

print(inspect.signature(minsearch.VectorSearch.fit))

import inspect

print(inspect.signature(minsearch.VectorSearch.search))
help(minsearch.VectorSearch)


(self, vectors, payload)
(self, query_vector, filter_dict=None, num_results=10, output_ids=False)
Help on class VectorSearch in module minsearch.vector:

class VectorSearch(builtins.object)
 |  VectorSearch(keyword_fields=None, numeric_fields=None, date_fields=None)
 |
 |  A vector search index using cosine similarity for vector search,
 |  exact matching for keyword fields, and range filters for numeric and date fields.
 |
 |  Takes a 2D numpy array of vectors and a list of payload documents, providing efficient
 |  similarity search with keyword, numeric, and date filtering capabilities.
 |
 |  Methods defined here:
 |
 |  __init__(self, keyword_fields=None, numeric_fields=None, date_fields=None)
 |      Initialize the VectorSearch index.
 |
 |      Args:
 |          keyword_fields (list, optional): List of keyword field names to index for exact matching. Defaults to empty list.
 |          numeric_fields (list, optional): List of numeric field names to index for range filters. Defau

In [47]:
# Create a MinSearch Vector Search index

# Create an empty Vector Search index

vector_index = minsearch.VectorSearch()

In [48]:
# Build the vector index

vector_index.fit(
    vectors=X,
    payload=chunks
)

In [49]:
# Query to search

query = "What metric do we use to evaluate a search engine?"

query_vector = embedder.encode(query)

In [50]:
print(query_vector)

[-3.45251120e-02 -4.76347907e-02 -9.52288143e-02  2.40308090e-02
 -1.80039094e-02  3.22336786e-02  4.11009181e-04  2.86892654e-02
  2.71596053e-02 -6.36475643e-02 -3.86707619e-02 -5.52097101e-02
  3.84640827e-02  3.63455416e-02 -9.32710316e-02 -5.29945155e-02
  8.20461574e-02 -5.03414745e-03  2.50910438e-02 -8.70656696e-02
  7.20100754e-02  1.29639035e-02  1.16451658e-01 -8.20982256e-02
 -6.52023347e-03 -4.00227584e-02 -8.36716391e-02 -1.24123525e-02
 -1.90502106e-02 -6.24077402e-03 -3.73876830e-02  6.12152591e-03
  5.60926979e-02  6.56479763e-02 -6.34850137e-02  3.43527274e-02
 -3.35941763e-02 -3.51476658e-02  2.54281298e-02 -9.75776093e-03
 -4.84886223e-02 -2.35025387e-02 -2.90664668e-02  3.65187041e-02
  2.23322831e-02 -3.29535923e-02 -4.82611746e-02  8.59540277e-03
 -3.70969933e-03  5.49367302e-02 -9.72623872e-02 -1.93188061e-04
 -1.25472844e-02 -2.15638746e-02 -5.50835867e-03  4.47906698e-02
 -5.21629946e-02 -2.24940001e-02 -2.30708396e-02 -8.72699683e-02
  9.89726366e-03 -5.08535

In [51]:
# Perform vector search

results = vector_index.search(
    query_vector,
    num_results=5
)

In [52]:
# Inspect the top result

results[0]

{'start': 0,
 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup

In [53]:
# Homework answer

results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

In [54]:
#Display the Top 5 results.

for rank, doc in enumerate(results, start=1):
    print(f"{rank}. {doc['filename']}")

1. 04-evaluation/lessons/05-search-metrics.md
2. 04-evaluation/lessons/01-intro.md
3. 01-agentic-rag/lessons/05-search.md
4. 04-evaluation/lessons/01-intro.md
5. 04-evaluation/lessons/15-next-steps.md


## Interpretation of Results

### Search Query

```
What metric do we use to evaluate a search engine?
```

### Top Search Result

```
04-evaluation/lessons/05-search-metrics.md
```

This lesson discusses the metrics used to evaluate the effectiveness of
information retrieval systems, making it the most semantically relevant
document for the given query.

---

### Top 5 Results

| Rank | Lesson | Interpretation |
|------|--------|----------------|
| 1 | 04-evaluation/lessons/05-search-metrics.md | Directly explains search evaluation metrics. |
| 2 | 04-evaluation/lessons/01-intro.md | Introduces the evaluation module. |
| 3 | 01-agentic-rag/lessons/05-search.md | Covers search fundamentals. |
| 4 | 04-evaluation/lessons/01-intro.md | Another highly relevant chunk from the introduction. |
| 5 | 04-evaluation/lessons/15-next-steps.md | Discusses evaluation in the broader workflow. |

Notice that multiple chunks originate from the same lesson page. This is
because the documents were chunked before indexing, allowing different
sections of the same lesson to be retrieved independently.

# From Manual Vector Search to MinSearch

In Question 3, we manually implemented a semantic search engine using
NumPy. This involved:

- generating embeddings for every document chunk,
- computing cosine similarity using the dot product,
- ranking all similarity scores,
- selecting the highest-scoring chunk.

Although this demonstrates the underlying mathematics, production systems
rarely implement vector search manually.

Instead, specialised libraries such as **MinSearch** encapsulate the same
workflow into a simple API.

The objective of this question is to perform exactly the same semantic
retrieval using MinSearch's `VectorSearch` class and compare it with the
manual implementation from Question 3.

## Key Learning

Question 4 demonstrates that the retrieval logic developed manually in
Question 3 can be replaced by a specialised vector search library.

Both approaches perform the same conceptual operations:

1. Convert the query into an embedding.
2. Compare the query embedding with all indexed document embeddings.
3. Compute cosine similarity.
4. Rank the results.
5. Return the most relevant document chunks.

The only difference is the level of abstraction.

| Question 3 | Question 4 |
|------------|------------|
| Manual implementation using NumPy | Library implementation using MinSearch |
| Explicit similarity computation | Similarity handled internally |
| Manual ranking | Automatic ranking |
| Educational implementation | Production-style implementation |

The retrieval algorithm remains unchanged; MinSearch simply provides a
cleaner, reusable interface.

# Conceptual Progression (Q1 → Q4)

# Progression of Concepts

The first four questions gradually build a semantic search engine.

### Q1 — Embeddings

Natural language is transformed into a numerical representation
(384-dimensional embedding).

```
Text
   │
   ▼
Embedding Model
   │
   ▼
Vector
```

---

### Q2 — Cosine Similarity

Two embeddings are compared to quantify their semantic similarity.

```
Query Vector
        │
        ▼
Document Vector
        │
        ▼
Cosine Similarity
```

---

### Q3 — Manual Vector Search

The query is compared against every document chunk.

```
Query
   │
   ▼
Embedding
   │
   ▼
295 Chunk Embeddings
   │
   ▼
Cosine Similarity
   │
   ▼
Ranking
   │
   ▼
Top Result
```

---

### Q4 — Vector Search Library

The same retrieval pipeline is implemented using MinSearch.

```
Query
   │
   ▼
Embedder
   │
   ▼
VectorSearch
   │
   ▼
Ranked Results
```

At this stage we have built a complete semantic search engine capable of
retrieving documents based on **meaning** rather than exact keyword
matching.

The next question compares this semantic retrieval approach with
traditional keyword-based text search.

# Evolution from embeddings (Q1) → semantic similarity (Q2) → manual vector search (Q3) → production-style vector search (Q4)

# Q5. Text Search vs Vector Search

## Objective

Until now, we have focused exclusively on semantic vector search.

However, traditional search engines rely primarily on **keyword matching**
rather than semantic similarity.

This question compares both retrieval strategies using the same document
collection and the same query.

The objective is to understand:

- when vector search performs better,
- when keyword search performs better,
- why modern Retrieval-Augmented Generation (RAG) systems often combine
  both approaches into **Hybrid Search**.

In [55]:
# Build a keyword search index using the chunk content
# Build the Text Index

text_index = minsearch.Index(
    text_fields=["content"]
)

text_index.fit(chunks)

# Define the Query

In [66]:
query = "How do I store vectors in PostgreSQL?"

In [67]:
print(query)

How do I store vectors in PostgreSQL?


# Vector Search

In [68]:
query_vector = embedder.encode(query)

vector_results = vector_index.search(
    query_vector,
    num_results=5
)

# Text Search

In [69]:
text_results = text_index.search(
    query,
    num_results=5
)

# Display Vector Results

In [70]:
print("Vector Search Results\n")

for i, doc in enumerate(vector_results, start=1):
    print(f"{i}. {doc['filename']}")

Vector Search Results

1. 02-vector-search/lessons/08-pgvector.md
2. 02-vector-search/lessons/08-pgvector.md
3. 03-orchestration/lessons/05-rag.md
4. 02-vector-search/lessons/08-pgvector.md
5. 02-vector-search/lessons/08-pgvector.md


# Display Text Results

In [71]:
print("Keyword Search Results\n")

for i, doc in enumerate(text_results, start=1):
    print(f"{i}. {doc['filename']}")

Keyword Search Results

1. 02-vector-search/lessons/02-embeddings.md
2. 03-orchestration/lessons/05-rag.md
3. 02-vector-search/lessons/01-intro.md
4. 03-orchestration/lessons/05-rag.md
5. 02-vector-search/lessons/01-intro.md


# Compare Both Result Sets

In [72]:
vector_files = {doc["filename"] for doc in vector_results}
text_files = {doc["filename"] for doc in text_results}

difference = vector_files - text_files

difference

{'02-vector-search/lessons/08-pgvector.md'}

In [73]:
list(difference)

['02-vector-search/lessons/08-pgvector.md']

## Interpretation of Results

### Search Query

```
How do I store vectors in PostgreSQL?
```

The query was executed using two independent retrieval methods:

- **Vector Search** (semantic similarity)
- **Keyword Search** (exact term matching)

The comparison identified one lesson that appeared only in the vector
search results.

### Unique Vector Search Result

```
02-vector-search/lessons/08-pgvector.md
```

This lesson discusses storing vector embeddings inside PostgreSQL using
the **PGVector** extension.

Although the wording of the lesson does not necessarily match the user's
query exactly, its semantic meaning is highly related.

Consequently, the embedding model successfully recognised the conceptual
relationship between the query and the lesson.

Keyword search, on the other hand, relies on exact word matching and did
not retrieve this lesson within its top-ranked results.

# Q5. Comparing Keyword Search and Vector Search

## Objective

So far, we have implemented semantic retrieval using vector embeddings.

However, traditional information retrieval systems primarily rely on
**keyword matching**.

This question compares two fundamentally different retrieval approaches
using the same document collection and the same user query.

The objective is to understand:

- how keyword search retrieves documents,
- how vector search retrieves documents,
- why they often return different results,
- why modern Retrieval-Augmented Generation (RAG) systems combine both
  approaches into Hybrid Search.

## Why Do the Results Differ?

Vector Search and Keyword Search use fundamentally different retrieval
strategies.

### Keyword Search

Keyword search compares words literally.

It searches for documents containing the same terms as the user's query.

Advantages:

- Excellent for identifiers, names and exact terminology.
- Fast and computationally efficient.

Limitations:

- Cannot understand synonyms.
- Cannot understand paraphrases.
- Misses semantically similar documents that use different wording.

---

### Vector Search

Vector search compares semantic meaning rather than individual words.

Both the query and every document are converted into embeddings.

Documents with similar meanings occupy nearby locations in the embedding
space and therefore receive higher cosine similarity scores.

Advantages:

- Understands semantic meaning.
- Handles paraphrases naturally.
- Finds conceptually related information.

Limitations:

- Requires embedding models.
- Requires vector indexing.
- Computationally more expensive than keyword search.

## Key Learning

Question 5 demonstrates that **neither retrieval strategy is universally
better**.

Each has its own strengths.

| Keyword Search | Vector Search |
|---------------|---------------|
| Exact word matching | Semantic matching |
| Excellent for identifiers and technical terms | Excellent for paraphrases and natural language |
| Fast | More computationally intensive |
| Misses synonyms | Understands semantic similarity |

Rather than replacing keyword search, vector search complements it.

Modern Retrieval-Augmented Generation (RAG) systems therefore combine
both methods using **Hybrid Search**, which will be implemented in the
next question using **Reciprocal Rank Fusion (RRF)**.

# Overall Progress (Q1 → Q5) 

# Progression of Concepts

The first five questions gradually build a complete semantic search
pipeline.

### Q1 — Embeddings

Convert natural language into numerical vectors.

```
Text
   │
   ▼
Embedding Model
   │
   ▼
384-dimensional Vector
```

---

### Q2 — Cosine Similarity

Measure semantic similarity between two embeddings.

```
Query Vector
        │
        ▼
Document Vector
        │
        ▼
Cosine Similarity
```

---

### Q3 — Manual Vector Search

Search every document chunk using cosine similarity.

```
Query
   │
   ▼
Embedding
   │
   ▼
295 Chunk Embeddings
   │
   ▼
Cosine Similarity
   │
   ▼
Ranking
```

---

### Q4 — Library-Based Vector Search

Replace the manual implementation with MinSearch.

```
Query
   │
   ▼
Embedding
   │
   ▼
VectorSearch Library
   │
   ▼
Top-k Results
```

---

### Q5 — Comparing Retrieval Strategies

Compare semantic retrieval with traditional keyword retrieval.

```
              Query
                 │
      ┌──────────┴──────────┐
      │                     │
      ▼                     ▼
Keyword Search        Vector Search
      │                     │
      ▼                     ▼
 Ranked Results      Ranked Results
      │                     │
      └──────────┬──────────┘
                 ▼
      Compare Retrieval Quality
```

This comparison demonstrates why neither method is sufficient on its own,
providing the motivation for **Hybrid Search**, which combines both
retrieval strategies.

# What I've learned conceptually

By the end of Q5, I've built an understanding that mirrors the evolution of retrieval systems:

# Q1: How text becomes embeddings.
# Q2: How semantic similarity is measured.
# Q3: How to implement vector search manually.
# Q4: How production libraries perform the same retrieval.
# Q5: Why semantic search and keyword search return different results and why combining them can improve retrieval.

This naturally sets the stage for Q6, where I'll implement Hybrid Search using Reciprocal Rank Fusion (RRF) to combine the strengths of both approaches.

# Transition: Q5 → Q6

# Q6. Hybrid Search using Reciprocal Rank Fusion (RRF)

## Objective

Question 5 demonstrated that keyword search and vector search often
retrieve different documents because they rely on different retrieval
strategies.

Rather than choosing one approach over the other, modern search systems
combine both methods to improve retrieval quality.

This approach is called **Hybrid Search**.

In this question, we use **Reciprocal Rank Fusion (RRF)** to merge the
ranked results returned by:

- Vector Search (semantic retrieval)
- Keyword Search (lexical retrieval)

The objective is to produce a single ranked list that benefits from the
strengths of both retrieval methods.

# Define the RRF function

In [74]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])

            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)

    return [docs[key] for key in ranked[:num_results]]

# New query

In [75]:
query = "How do I give the model access to tools?"

print(query)

How do I give the model access to tools?


# Vector Search

In [77]:
query_vector = embedder.encode(query)

vector_results = vector_index.search(
    query_vector,
    num_results=5
)

print(vector_results)

[{'start': 2000, 'content': 'wrong.\n\n## The project\n\nRAG solves these problems by giving the LLM relevant documents at\nquestion time. We don\'t hope the model memorized the answer. We\nretrieve the right information and hand it to the LLM, and the model\ngenerates a grounded response. This lets us inject knowledge the model\nnever saw during training. That\'s why RAG is still the most common way\npeople use LLMs in the industry.\n\nTo make this concrete, we build a FAQ agent for our course. A student\nasks something like "when does the course start?" and the agent answers\nfrom the FAQ data we prepared.\n\nThis module has two parts.\n\nIn Part 1 (the next 9 lessons) we will:\n\n- Understand what RAG is and how it works\n- Build a search engine over a real FAQ dataset\n- Write a prompt that combines the user\'s question with search results\n- Wire it all together into a working RAG pipeline\n- Split ingestion and query into separate processes\n\nIn Part 2, we make the pipeline agen

# Keyword Search

In [78]:
text_results = text_index.search(
    query,
    num_results=5
)

print(text_results)

[{'start': 0, 'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can c

# Apply Hybrid Search

In [80]:
results = rrf(
    [vector_results, text_results]
)

print(results)

[{'start': 4000, 'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function ca

# Homework answer

In [81]:
results[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'

# Display the hybrid ranking

In [82]:
for i, doc in enumerate(results, start=1):
    print(f"{i}. {doc['filename']}")

1. 01-agentic-rag/lessons/13-function-calling.md
2. 01-agentic-rag/lessons/01-intro.md
3. 01-agentic-rag/lessons/14-agentic-loop.md
4. 04-evaluation/lessons/02-ground-truth.md
5. 01-agentic-rag/lessons/16-other-frameworks.md


## Interpretation of Results

Hybrid Search combines the strengths of both retrieval strategies.

Instead of relying solely on semantic similarity or exact keyword
matching, Reciprocal Rank Fusion (RRF) merges the ranked outputs of both
search methods.

Unlike score averaging, RRF ignores the raw similarity scores because
vector similarity scores and keyword relevance scores are measured on
different scales.

Instead, RRF considers only the **rank position** of each document.

Documents that consistently appear near the top of multiple ranked lists
receive the highest combined score.

Consequently, a document that performs well in both retrieval methods can
outrank a document that appears first in only one search method.

This produces a more robust ranking than either search strategy alone.

## How Reciprocal Rank Fusion Works

Suppose two search engines return the following rankings.

### Vector Search

| Rank | Document |
|------|----------|
| 0 | A |
| 1 | B |
| 2 | C |

### Keyword Search

| Rank | Document |
|------|----------|
| 0 | B |
| 1 | A |
| 2 | D |

Rather than comparing similarity scores, RRF assigns each document a
score according to its ranking position.

```
Rank 0  → 1 / (60 + 0)

Rank 1  → 1 / (60 + 1)

Rank 2  → 1 / (60 + 2)
```

The contributions from all ranked lists are added together.

A document appearing near the top of both rankings accumulates a higher
total score than one appearing only once.

This makes RRF simple, robust and independent of the scoring methods
used by the individual retrieval systems.

## Key Learning

Question 6 demonstrates why Hybrid Search has become the dominant
retrieval strategy for Retrieval-Augmented Generation (RAG) systems.

Neither keyword search nor vector search is universally superior.

Keyword search excels at matching exact terminology, identifiers and
technical names.

Vector search excels at understanding semantic meaning, paraphrases and
natural language queries.

Hybrid Search combines both approaches, producing more reliable retrieval
performance than either method individually.

For this reason, most modern enterprise search systems and production
RAG applications employ Hybrid Search together with reranking models.

# Final Summary of Homework 2

# Homework 2 Summary

This homework gradually built a complete semantic retrieval system.

## Q1 — Embeddings

Convert natural language into numerical vectors.

```
Text
   │
   ▼
Embedding Model
```

---

## Q2 — Cosine Similarity

Compare two embeddings.

```
Embedding A

↓

Embedding B

↓

Cosine Similarity
```

---

## Q3 — Manual Vector Search

```
Query

↓

Embedding

↓

295 Chunk Embeddings

↓

Cosine Similarity

↓

Ranking
```

---

## Q4 — Vector Search Library

```
Query

↓

Embedding

↓

MinSearch VectorSearch

↓

Top Results
```

---

## Q5 — Comparing Retrieval Methods

```
                 Query
                    │
        ┌───────────┴───────────┐
        ▼                       ▼
 Keyword Search          Vector Search
```

---

## Q6 — Hybrid Search

```
                 Query
                    │
        ┌───────────┴───────────┐
        ▼                       ▼
 Keyword Search          Vector Search
        │                       │
        └───────────┬───────────┘
                    ▼
      Reciprocal Rank Fusion (RRF)
                    │
                    ▼
          Final Ranked Documents
```

By the end of this homework, we have progressed from generating
embeddings to implementing a complete hybrid retrieval pipeline using
semantic search, keyword search and Reciprocal Rank Fusion.

# Homework 2 Answers
Question	Correct Answer
Q1	-0.02
Q2	0.37
Q3	02-vector-search/lessons/07-sqlitesearch-vector.md
Q4	04-evaluation/lessons/05-search-metrics.md
Q5	02-vector-search/lessons/08-pgvector.md
Q6	01-agentic-rag/lessons/13-function-calling.md

# I've completed Module 2 by implementing:

# 1. Sentence embeddings with an ONNX model,
# 2. Cosine similarity,
# 3. Manual vector search,
# 4. Library-based vector search with MinSearch,
# 5. A comparison of keyword vs. semantic retrieval,
# 6. And finally, Hybrid Search using Reciprocal Rank Fusion (RRF)—a technique widely used in production RAG systems